# LeetCode 1053 — Previous Permutation With One Swap bằng Python

Link đề: https://leetcode.com/problems/previous-permutation-with-one-swap/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Greedy (tham lam)

Skeleton LeetCode:

```python
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        ...
```

Ghi chú: cho mảng số nguyên dương `arr` (có thể trùng nhau). Hãy hoán vị mảng bằng **đúng một lần đổi chỗ hai phần tử** sao cho kết quả **nhỏ hơn** `arr` theo thứ tự từ điển (lexicographic) nhưng **lớn nhất có thể**. Nếu không làm được thì trả về chính `arr`. Đây là bài **tham lam**: vị trí đổi đầu tiên càng **xa về bên phải** càng tốt, rồi chọn giá trị thay vào lớn nhất mà vẫn nhỏ hơn, và với các giá trị trùng thì chọn vị trí **bên trái nhất**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **thứ tự từ điển** của danh sách và vì sao "đổi càng về bên phải thì kết quả càng lớn".
2. Nắm các công cụ Python cần thiết: so sánh danh sách bằng `<`, đổi chỗ bằng `a[i], a[j] = a[j], a[i]`, sao chép bằng `arr[:]`, `itertools.combinations`, `bisect`.
3. Hiểu cấu trúc **hậu tố không giảm**: sau vị trí `i` phải đổi, phần còn lại đã được sắp xếp không giảm, nên chọn `j` rất dễ.
4. Đi qua 4 cách: thử mọi cặp `(i, j)` (brute force, dùng làm oracle), tham lam dùng mảng "min hậu tố", tham lam một lượt quét (cách nộp chính) và dùng `bisect` trên hậu tố không giảm.
5. Tránh các lỗi kinh điển: chọn `i` đầu tiên thay vì cuối cùng, chọn `j` là giá trị nhỏ nhất, bỏ sót quy tắc "trái nhất" khi có phần tử trùng, đổi cho phần tử bằng nhau, trả về `[]` khi không làm được và quên rằng hàm sửa mảng **tại chỗ**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `arr` (không nhất thiết khác nhau). Trả về hoán vị **lớn nhất theo thứ tự từ điển mà nhỏ hơn `arr`**, tạo ra bằng **đúng một lần đổi chỗ**. Nếu không thể, trả về chính mảng `arr`.

Một lần đổi chỗ (swap) là đổi chỗ hai phần tử `arr[i]` và `arr[j]`.

**Ví dụ 1:** `arr = [3,2,1]` -> `[3,1,2]` (đổi `2` và `1`).

**Ví dụ 2:** `arr = [1,1,5]` -> `[1,1,5]` (đã là hoán vị nhỏ nhất).

**Ví dụ 3:** `arr = [1,9,4,6,7]` -> `[1,7,4,6,9]` (đổi `9` và `7`).

**Ràng buộc:**

- `1 <= arr.length <= 10^4`
- `1 <= arr[i] <= 10^4`

In [1]:
from typing import List
from itertools import combinations, permutations, product
from bisect import bisect_left
import random, time

examples = [
    ([3, 2, 1], [3, 1, 2]),
    ([1, 1, 5], [1, 1, 5]),
    ([1, 9, 4, 6, 7], [1, 7, 4, 6, 9]),
]
for arr, expected in examples:
    print(f"arr={arr} -> kỳ vọng {expected}")

arr=[3, 2, 1] -> kỳ vọng [3, 1, 2]
arr=[1, 1, 5] -> kỳ vọng [1, 1, 5]
arr=[1, 9, 4, 6, 7] -> kỳ vọng [1, 7, 4, 6, 9]


## 3. Kiến thức nền

### 3.1. Thứ tự từ điển của danh sách

Python so sánh hai danh sách **từ trái sang phải**: gặp vị trí đầu tiên khác nhau thì phần tử nào nhỏ hơn quyết định danh sách đó nhỏ hơn. Vì vậy **vị trí khác nhau đầu tiên** quan trọng nhất.

In [2]:
print([1, 7, 4, 6, 9] < [1, 9, 4, 6, 7])      # khác đầu tiên ở vị trí 1: 7 < 9
print([3, 1, 2] < [3, 2, 1])                  # khác đầu tiên ở vị trí 1: 1 < 2
print([1, 2, 9] < [1, 3, 0])                  # vị trí 1 quyết định, các vị trí sau không còn quan trọng
print([1, 1, 5] < [1, 1, 5])                  # bằng nhau: không nhỏ hơn

True
True
True
False


### 3.2. Đổi chỗ hai phần tử và sao chép

`a[i], a[j] = a[j], a[i]` đổi chỗ **tại chỗ** (sửa chính danh sách `a`). Muốn giữ bản gốc phải sao chép trước bằng `a[:]` hoặc `list(a)`; gán `b = a` chỉ tạo thêm **tên khác** cho cùng một danh sách.

In [3]:
a = [3, 2, 1]
b = a                      # cùng một danh sách
c = a[:]                   # bản sao
a[1], a[2] = a[2], a[1]
print("a =", a, "| b =", b, "| c =", c)

a = [3, 1, 2] | b = [3, 1, 2] | c = [3, 2, 1]


### 3.3. Hậu tố không giảm và `bisect_left`

Nếu một đoạn đã **không giảm** (sắp xếp tăng dần, cho phép trùng) thì `bisect_left(đoạn, x)` trả về chỉ số đầu tiên có giá trị `>= x` bằng tìm kiếm nhị phân `O(log n)`. Tham số `lo`, `hi` giới hạn đoạn `[lo, hi)`.

In [4]:
s = [2, 2, 4, 4, 4, 7]
print(bisect_left(s, 4), bisect_left(s, 5), bisect_left(s, 1), bisect_left(s, 8))
arr = [1, 9, 4, 6, 7]
print("trong arr[2:5] =", arr[2:5], "bisect_left(arr, 6, 2, 5) =", bisect_left(arr, 6, 2, 5))

2 5 0 6
trong arr[2:5] = [4, 6, 7] bisect_left(arr, 6, 2, 5) = 3


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `arr = [1,9,4,6,7]`.

- Muốn nhỏ hơn `arr`, ở **vị trí đổi đầu tiên** `i` ta phải đưa vào một giá trị **nhỏ hơn** `arr[i]` từ bên phải.
- Để kết quả lớn nhất, vị trí `i` nên **càng về cuối càng tốt** (giữ nguyên tiền tố dài nhất).
- Quét từ phải sang trái tìm `i` đầu tiên thỏa `arr[i] > arr[i + 1]`: `arr[3] = 6 <= arr[4] = 7` bỏ qua, `arr[2] = 4 <= arr[3] = 6` bỏ qua, `arr[1] = 9 > arr[2] = 4` dừng. Vậy `i = 1`.
- Phần bên phải `[4, 6, 7]` không giảm. Giá trị lớn nhất **nhỏ hơn** `9` trong đó là `7` (vị trí `4`).
- Đổi: `[1, 7, 4, 6, 9]` ✓.

Ví dụ 2: `[1,1,5]` không có `i` nào thỏa `arr[i] > arr[i+1]` (mảng không giảm = đã nhỏ nhất) ⇒ trả về chính nó.

Ví dụ có phần tử trùng: `[3,1,1]`. `i = 0` (`3 > 1`); các giá trị `< 3` bên phải là `1, 1`. Đổi với `1` ở vị trí `1` cho `[1,3,1]`; đổi với `1` ở vị trí `2` cho `[1,1,3]`. Kết quả đầu **lớn hơn**, nên chọn vị trí **bên trái nhất** trong các giá trị trùng.

## 5. Cách 1 — Brute Force: thử mọi cặp `(i, j)`

**Ý tưởng.** Với mọi `i < j`, đổi chỗ trên một bản sao, nếu bản sao **nhỏ hơn** `arr` thì ứng viên; giữ ứng viên lớn nhất. Nếu không có ứng viên thì trả về `arr`. Đúng định nghĩa đề nên là **oracle độc lập**.

In [5]:
def prev_brute(arr: List[int]) -> List[int]:
    n = len(arr)
    best = None
    for i in range(n):
        for j in range(i + 1, n):
            b = arr[:]
            b[i], b[j] = b[j], b[i]
            if b < arr and (best is None or b > best):
                best = b
    return best if best is not None else arr[:]

for arr, expected in examples:
    got = prev_brute(arr)
    print(f"brute({arr}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute([3, 2, 1]) = [3, 1, 2] OK
brute([1, 1, 5]) = [1, 1, 5] OK
brute([1, 9, 4, 6, 7]) = [1, 7, 4, 6, 9] OK


**Độ phức tạp và nút thắt.** Có `n(n - 1) / 2` cặp, mỗi cặp tốn `O(n)` để sao chép và so sánh ⇒ `O(n^3)`. Với `n = 10^4` là `~ 10^12`: không thể. Đo thử:

In [6]:
rng = random.Random(1053)
for n in (50, 100, 200, 400):
    arr = [rng.randint(1, 10 ** 4) for _ in range(n)]
    t0 = time.perf_counter()
    prev_brute(arr)
    dt = time.perf_counter() - t0
    print(f"n={n:4}: {dt * 1000:9.1f} ms -> ngoại suy n=10^4: ~{dt * (10 ** 4 / n) ** 3 / 3600:10.1f} giờ")

n=  50:       0.4 ms -> ngoại suy n=10^4: ~       0.8 giờ
n= 100:       1.6 ms -> ngoại suy n=10^4: ~       0.4 giờ
n= 200:      11.8 ms -> ngoại suy n=10^4: ~       0.4 giờ
n= 400:     134.6 ms -> ngoại suy n=10^4: ~       0.6 giờ


Nút thắt: ta thử cả những cặp chắc chắn không tối ưu. Nếu biết **vị trí `i` tốt nhất** và **giá trị `j` tốt nhất**, chỉ cần một lần quét.

## 6. Ý tưởng thuật toán / bước đột phá

**Ba quy tắc tham lam.**

1. **Chọn `i` lớn nhất** (xa bên phải nhất) sao cho tồn tại `j > i` với `arr[j] < arr[i]`. Lý do: so sánh hai ứng viên có vị trí đổi đầu tiên `i1 < i2`; tại `i1` ứng viên thứ nhất bị **giảm** còn ứng viên thứ hai vẫn bằng `arr[i1]`, nên ứng viên thứ hai lớn hơn.
2. Với `i` đã chọn, **chọn `j` có giá trị lớn nhất mà vẫn `< arr[i]`** (vị trí `i` nhận giá trị lớn nhất có thể mà vẫn nhỏ hơn).
3. Nếu giá trị đó xuất hiện nhiều lần, chọn **`j` bên trái nhất**: đặt `arr[i]` (lớn hơn giá trị đó) vào vị trí càng sớm thì kết quả càng lớn.

**Cấu trúc đặc biệt.** `i` là vị trí **cuối cùng** có `arr[i] > arr[i + 1]`. Khi đó phần sau `i` không giảm, nên giá trị lớn nhất `< arr[i]` nằm ngay trước phần tử đầu tiên `>= arr[i]`.

| Cách | Ý tưởng |
|---|---|
| 2 | Dùng mảng `suffix_min`: `i` là vị trí cuối cùng có `arr[i] > suffix_min[i + 1]`; quét tìm `j` |
| 3 | Một lượt quét phải-trái tìm `i`, rồi `j` từ phải sang trái (nộp) |
| 4 | Tìm `i` như cách 3; tìm `j` bằng `bisect_left` trên hậu tố không giảm |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mảng min hậu tố

**Giải thích từng đoạn.**

- `suffix_min[k]` = giá trị nhỏ nhất của `arr[k:]` (và `+inf` ngoài mảng).
- Duyệt `i` từ phải sang trái; vị trí đầu tiên (cuối cùng theo chiều trái-phải) thỏa `arr[i] > suffix_min[i + 1]` là `i` cần tìm.
- Quét `j` trong `(i, n)`: lấy giá trị lớn nhất `< arr[i]`; khi hòa giá trị chọn `j` nhỏ nhất (dùng điều kiện `>` chặt để không ghi đè).

In [7]:
def prev_suffix_min(arr: List[int]) -> List[int]:
    n = len(arr)
    arr = arr[:]
    suffix_min = [float("inf")] * (n + 1)
    for k in range(n - 1, -1, -1):
        suffix_min[k] = min(arr[k], suffix_min[k + 1])
    for i in range(n - 1, -1, -1):
        if arr[i] > suffix_min[i + 1]:
            best_j = -1
            for j in range(i + 1, n):
                if arr[j] < arr[i] and (best_j == -1 or arr[j] > arr[best_j]):   # chặt: hòa thì giữ j bên trái
                    best_j = j
            arr[i], arr[best_j] = arr[best_j], arr[i]
            return arr
    return arr

for arr, expected in examples:
    got = prev_suffix_min(arr)
    print(f"suffix_min({arr}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

suffix_min([3, 2, 1]) = [3, 1, 2] OK
suffix_min([1, 1, 5]) = [1, 1, 5] OK
suffix_min([1, 9, 4, 6, 7]) = [1, 7, 4, 6, 9] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        n = len(arr)
        suffix_min = [float("inf")] * (n + 1)            # suffix_min[k] = min(arr[k:])
        for k in range(n - 1, -1, -1):
            suffix_min[k] = min(arr[k], suffix_min[k + 1])
        for i in range(n - 1, -1, -1):
            if arr[i] > suffix_min[i + 1]:               # tồn tại phần tử nhỏ hơn bên phải
                best_j = -1
                for j in range(i + 1, n):
                    if arr[j] < arr[i] and (best_j == -1 or arr[j] > arr[best_j]):
                        best_j = j                       # giá trị lớn nhất < arr[i]; hòa thì giữ j bên trái
                arr[i], arr[best_j] = arr[best_j], arr[i]
                return arr
        return arr

sol_min = lambda a: Solution().prevPermOpt1(a[:])
for arr, expected in examples:
    assert sol_min(arr) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)` (quét `j` chỉ xảy ra một lần), bộ nhớ `O(n)` cho `suffix_min`.

---

### 7.2. Cách 3 — Một lượt quét, không cần mảng phụ (cách nộp chính)

**Giải thích từng đoạn.**

- `i` chạy từ `n - 2` giảm dần tới khi `arr[i] > arr[i + 1]`. Nếu `i < 0`: mảng không giảm ⇒ đã là hoán vị nhỏ nhất ⇒ trả về nguyên mảng.
- Phần sau `i` không giảm. Tìm `j` từ **cuối** mảng: lùi tới khi `arr[j] < arr[i]`; đây là phần tử **lớn nhất** `< arr[i]` (vì đang đi từ giá trị lớn xuống).
- Nếu bên trái `j` còn các giá trị **bằng** `arr[j]` thì lùi tiếp để lấy bản trái nhất.
- Đổi chỗ và trả về.

In [9]:
def prev_scan(arr: List[int]) -> List[int]:
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    j = n - 1
    while arr[j] >= arr[i]:
        j -= 1
    while arr[j - 1] == arr[j]:
        j -= 1
    arr[i], arr[j] = arr[j], arr[i]
    return arr

for arr, expected in examples:
    got = prev_scan(arr)
    print(f"scan({arr}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

scan([3, 2, 1]) = [3, 1, 2] OK
scan([1, 1, 5]) = [1, 1, 5] OK
scan([1, 9, 4, 6, 7]) = [1, 7, 4, 6, 9] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        n = len(arr)
        i = n - 2
        while i >= 0 and arr[i] <= arr[i + 1]:           # tìm vị trí cuối cùng có arr[i] > arr[i + 1]
            i -= 1
        if i < 0:
            return arr                                   # mảng không giảm: đã nhỏ nhất
        j = n - 1
        while arr[j] >= arr[i]:                          # phần tử cuối cùng (lớn nhất trong hậu tố) < arr[i]
            j -= 1
        while arr[j - 1] == arr[j]:                      # nhiều giá trị trùng: lấy bản bên trái nhất
            j -= 1
        arr[i], arr[j] = arr[j], arr[i]
        return arr

sol_scan = lambda a: Solution().prevPermOpt1(a[:])
for arr, expected in examples:
    assert sol_scan(arr) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(1)` thêm (sửa tại chỗ).

---

### 7.3. Cách 4 — `bisect_left` trên hậu tố không giảm

**Ý tưởng khác.** Sau khi có `i`, hậu tố `arr[i + 1:]` không giảm. Vị trí `p = bisect_left(arr, arr[i], i + 1, n)` là phần tử đầu tiên `>= arr[i]`, nên `arr[p - 1]` là giá trị lớn nhất `< arr[i]`. Gọi `v = arr[p - 1]`; bản trái nhất của `v` là `bisect_left(arr, v, i + 1, n)`. Không cần vòng `while` lùi.

In [11]:
def prev_bisect(arr: List[int]) -> List[int]:
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    p = bisect_left(arr, arr[i], i + 1, n)          # phần tử đầu tiên >= arr[i]
    v = arr[p - 1]                                  # giá trị lớn nhất < arr[i]
    j = bisect_left(arr, v, i + 1, n)               # bản trái nhất của v
    arr[i], arr[j] = arr[j], arr[i]
    return arr

for arr, expected in examples:
    got = prev_bisect(arr)
    print(f"bisect({arr}) = {got}", "OK" if got == expected else "SAI")
    assert got == expected

bisect([3, 2, 1]) = [3, 1, 2] OK
bisect([1, 1, 5]) = [1, 1, 5] OK
bisect([1, 9, 4, 6, 7]) = [1, 7, 4, 6, 9] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        from bisect import bisect_left
        n = len(arr)
        i = n - 2
        while i >= 0 and arr[i] <= arr[i + 1]:
            i -= 1
        if i < 0:
            return arr
        p = bisect_left(arr, arr[i], i + 1, n)           # phần tử đầu tiên >= arr[i] trong hậu tố không giảm
        v = arr[p - 1]                                   # giá trị lớn nhất < arr[i]
        j = bisect_left(arr, v, i + 1, n)                # bản trái nhất của v
        arr[i], arr[j] = arr[j], arr[i]
        return arr

sol_bisect = lambda a: Solution().prevPermOpt1(a[:])
for arr, expected in examples:
    assert sol_bisect(arr) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** tìm `i` là `O(n)`, tìm `j` là `O(log n)`; tổng `O(n)`, bộ nhớ `O(1)` thêm.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `[1, 9, 4, 6, 7]`

In từng bước quét `i`, rồi `j`, rồi đổi chỗ.

In [13]:
def trace_scan(arr):
    arr = arr[:]
    n = len(arr)
    print("arr ban đầu:", arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        print(f"  i={i}: arr[i]={arr[i]} <= arr[i+1]={arr[i + 1]} -> lùi i")
        i -= 1
    if i < 0:
        print("  không có arr[i] > arr[i+1]: mảng không giảm, trả về nguyên mảng")
        return arr
    print(f"  dừng ở i={i}: arr[i]={arr[i]} > arr[i+1]={arr[i + 1]}; hậu tố = {arr[i + 1:]}")
    j = n - 1
    while arr[j] >= arr[i]:
        print(f"  j={j}: arr[j]={arr[j]} >= arr[i]={arr[i]} -> lùi j")
        j -= 1
    print(f"  j={j}: arr[j]={arr[j]} < arr[i]={arr[i]} -> giá trị lớn nhất nhỏ hơn arr[i]")
    while arr[j - 1] == arr[j]:
        print(f"  arr[j-1] == arr[j] == {arr[j]} -> lùi j để lấy bản bên trái")
        j -= 1
    arr[i], arr[j] = arr[j], arr[i]
    print(f"  đổi arr[{i}] và arr[{j}] -> {arr}")
    return arr

assert trace_scan([1, 9, 4, 6, 7]) == [1, 7, 4, 6, 9]

arr ban đầu: [1, 9, 4, 6, 7]
  i=3: arr[i]=6 <= arr[i+1]=7 -> lùi i
  i=2: arr[i]=4 <= arr[i+1]=6 -> lùi i
  dừng ở i=1: arr[i]=9 > arr[i+1]=4; hậu tố = [4, 6, 7]
  j=4: arr[j]=7 < arr[i]=9 -> giá trị lớn nhất nhỏ hơn arr[i]
  đổi arr[1] và arr[4] -> [1, 7, 4, 6, 9]


### 8.2. Trường hợp có phần tử trùng và trường hợp không làm được

Xem vì sao phải lấy bản trái nhất.

In [14]:
assert trace_scan([5, 1, 3, 3, 3]) == [3, 1, 5, 3, 3]
print()
assert trace_scan([3, 1, 1]) == [1, 3, 1]
print()
assert trace_scan([1, 1, 5]) == [1, 1, 5]

arr ban đầu: [5, 1, 3, 3, 3]
  i=3: arr[i]=3 <= arr[i+1]=3 -> lùi i
  i=2: arr[i]=3 <= arr[i+1]=3 -> lùi i
  i=1: arr[i]=1 <= arr[i+1]=3 -> lùi i
  dừng ở i=0: arr[i]=5 > arr[i+1]=1; hậu tố = [1, 3, 3, 3]
  j=4: arr[j]=3 < arr[i]=5 -> giá trị lớn nhất nhỏ hơn arr[i]
  arr[j-1] == arr[j] == 3 -> lùi j để lấy bản bên trái
  arr[j-1] == arr[j] == 3 -> lùi j để lấy bản bên trái
  đổi arr[0] và arr[2] -> [3, 1, 5, 3, 3]

arr ban đầu: [3, 1, 1]
  i=1: arr[i]=1 <= arr[i+1]=1 -> lùi i
  dừng ở i=0: arr[i]=3 > arr[i+1]=1; hậu tố = [1, 1]
  j=2: arr[j]=1 < arr[i]=3 -> giá trị lớn nhất nhỏ hơn arr[i]
  arr[j-1] == arr[j] == 1 -> lùi j để lấy bản bên trái
  đổi arr[0] và arr[1] -> [1, 3, 1]

arr ban đầu: [1, 1, 5]
  i=1: arr[i]=1 <= arr[i+1]=5 -> lùi i
  i=0: arr[i]=1 <= arr[i+1]=1 -> lùi i
  không có arr[i] > arr[i+1]: mảng không giảm, trả về nguyên mảng


### 8.3. Vì sao `i` là vị trí cuối cùng có `arr[i] > arr[i + 1]`

Sau `i` mảng không giảm nên không phần tử nào ở sau còn có "phần tử nhỏ hơn bên phải". Ta in `i` tìm được bằng hai định nghĩa khác nhau và so sánh.

In [15]:
def last_i_by_scan(arr):
    i = len(arr) - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    return i

def last_i_by_definition(arr):
    best = -1
    for i in range(len(arr)):
        if any(arr[j] < arr[i] for j in range(i + 1, len(arr))):
            best = i
    return best

rng = random.Random(83)
for _ in range(2000):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 9))]
    assert last_i_by_scan(a) == last_i_by_definition(a), a
print("hai cách tìm i trùng nhau trên 2000 mảng ngẫu nhiên")

hai cách tìm i trùng nhau trên 2000 mảng ngẫu nhiên


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (một lượt quét): `O(n)` thời gian, `O(1)` bộ nhớ thêm, rất ngắn.

```python
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        n = len(arr)
        i = n - 2
        while i >= 0 and arr[i] <= arr[i + 1]:           # tìm vị trí cuối cùng có arr[i] > arr[i + 1]
            i -= 1
        if i < 0:
            return arr                                   # mảng không giảm: đã nhỏ nhất
        j = n - 1
        while arr[j] >= arr[i]:                          # phần tử cuối cùng (lớn nhất trong hậu tố) < arr[i]
            j -= 1
        while arr[j - 1] == arr[j]:                      # nhiều giá trị trùng: lấy bản bên trái nhất
            j -= 1
        arr[i], arr[j] = arr[j], arr[i]
        return arr
```

In [16]:
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        n = len(arr)
        i = n - 2
        while i >= 0 and arr[i] <= arr[i + 1]:           # tìm vị trí cuối cùng có arr[i] > arr[i + 1]
            i -= 1
        if i < 0:
            return arr                                   # mảng không giảm: đã nhỏ nhất
        j = n - 1
        while arr[j] >= arr[i]:                          # phần tử cuối cùng (lớn nhất trong hậu tố) < arr[i]
            j -= 1
        while arr[j - 1] == arr[j]:                      # nhiều giá trị trùng: lấy bản bên trái nhất
            j -= 1
        arr[i], arr[j] = arr[j], arr[i]
        return arr

sol = lambda a: Solution().prevPermOpt1(a[:])            # truyền bản sao vì hàm sửa mảng tại chỗ
for arr, expected in examples:
    got = sol(arr)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

[3, 1, 2] OK
[1, 1, 5] OK
[1, 7, 4, 6, 9] OK


## 10. Vì sao đúng

Gọi `R` là tập các cặp `(i, j)`, `i < j`, với `arr[j] < arr[i]`. Một lần đổi cho kết quả **nhỏ hơn** `arr` khi và chỉ khi cặp đó thuộc `R` (vị trí khác nhau đầu tiên là `i`, và tại đó giá trị mới `arr[j]` nhỏ hơn `arr[i]`). Đổi hai phần tử bằng nhau không đổi mảng; đổi với `arr[j] > arr[i]` làm mảng lớn hơn. Nếu `R` rỗng (mọi phần tử đều không lớn hơn mọi phần tử sau nó, tức mảng không giảm) thì trả về `arr`.

**Quy tắc 1 (`i` lớn nhất).** Hai cặp `(i1, j1)`, `(i2, j2)` thuộc `R` với `i1 < i2`: kết quả 1 khác `arr` đầu tiên ở `i1` (nhỏ hơn `arr[i1]`), còn kết quả 2 trùng `arr` tại mọi vị trí `< i2`, nên tại `i1` có giá trị `arr[i1]` lớn hơn. Vậy kết quả 2 lớn hơn. Do đó vị trí `i` tối ưu là **`i` lớn nhất** xuất hiện trong `R`.

**Cấu trúc.** Gọi `i` là vị trí cuối cùng có `arr[i] > arr[i + 1]`. Với mọi `k > i`: `arr[k] <= arr[k + 1]`, nên hậu tố sau `i` không giảm. Với `i' > i` mọi phần tử sau `arr[i']` đều `>= arr[i']`, nên `i'` không thuộc `R`. Ngược lại `(i, i + 1) ∈ R`, nên `i` là `i` lớn nhất trong `R`. (Nếu chỉ tồn tại `i'` lớn hơn có phần tử nhỏ hơn ở xa, ta lập luận: `arr[i'] <= arr[i' + 1]` mà `arr[j] < arr[i']`, `j > i' + 1` thì `i' + 1` cũng thuộc `R`, mâu thuẫn tính lớn nhất; kiểm bằng máy ở mục 8.3.)

**Quy tắc 2 (giá trị `j`).** Với `i` cố định, các kết quả giống nhau ở các vị trí `< i`, nên kết quả lớn nhất là kết quả có `arr[i]` mới **lớn nhất**: chọn `v` lớn nhất trong hậu tố thỏa `v < arr[i]`. Hậu tố không giảm nên đó là phần tử cuối cùng `< arr[i]` (cách 3 lùi `j` từ cuối; cách 4 dùng `bisect_left`).

**Quy tắc 3 (trái nhất).** Các vị trí `j1 < j2` cùng giá trị `v`: kết quả 1 có `arr[i]` ở `j1`, kết quả 2 có `v` ở `j1` (các vị trí trước `j1` giống nhau). Vì `v < arr[i]` nên kết quả 1 lớn hơn. Chọn `j` trái nhất. Vòng `while arr[j - 1] == arr[j]` thực hiện đúng điều này và không vượt qua `i` vì `arr[i] != arr[j]`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) thử mọi cặp. Ta đối chiếu **mọi** mảng độ dài `1..7` với giá trị `1..4` (tổng `4^1 + ... + 4^7 = 21844` mảng) giữa mọi cách và oracle.

In [17]:
total = 0
for n in range(1, 8):
    for tup in product(range(1, 5), repeat=n):
        arr = list(tup)
        ref = prev_brute(arr)
        assert prev_suffix_min(arr) == prev_scan(arr) == prev_bisect(arr) == sol(arr) == sol_min(arr) == sol_scan(arr) == sol_bisect(arr) == ref, arr
        total += 1
print("đã đối chiếu", total, "mảng nhỏ: mọi cách khớp oracle")

đã đối chiếu 21844 mảng nhỏ: mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cặp | `O(n^3)` | `O(n)` |
| 2. Min hậu tố | `O(n)` | `O(n)` |
| 3. Một lượt quét | `O(n)` | `O(1)` |
| 4. `bisect_left` | `O(n)` | `O(1)` |

Đo ở cỡ tối đa (`n = 10^4`) với các dạng: ngẫu nhiên, giảm dần, không giảm, toàn bằng nhau, nhiều giá trị trùng:

In [18]:
rng = random.Random(11)
n = 10 ** 4
cases = {
    "ngẫu nhiên": [rng.randint(1, 10 ** 4) for _ in range(n)],
    "giảm dần": list(range(n, 0, -1)),
    "không giảm": sorted(rng.randint(1, 10 ** 4) for _ in range(n)),
    "toàn bằng nhau": [7] * n,
    "ít giá trị (1..3)": [rng.randint(1, 3) for _ in range(n)],
    "đuôi không giảm dài": [10 ** 4] + sorted(rng.randint(1, 9999) for _ in range(n - 1)),
}
for label, arr in cases.items():
    ref = prev_scan(arr)
    row = [f"{label:20}"]
    for name, fn in (("min hậu tố", sol_min), ("quét", sol_scan), ("bisect", sol_bisect), ("nộp", sol)):
        t0 = time.perf_counter()
        got = fn(arr)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:6.2f} ms")
    print(" | ".join(row))

ngẫu nhiên           | min hậu tố   0.10 ms | quét   0.08 ms | bisect   0.04 ms | nộp   0.04 ms
giảm dần             | min hậu tố   0.04 ms | quét   0.03 ms | bisect   0.03 ms | nộp   0.03 ms
không giảm           | min hậu tố   1.04 ms | quét   1.04 ms | bisect   1.04 ms | nộp   1.19 ms
toàn bằng nhau       | min hậu tố   1.06 ms | quét   1.01 ms | bisect   1.03 ms | nộp   1.02 ms
ít giá trị (1..3)    | min hậu tố   0.03 ms | quét   0.02 ms | bisect   0.02 ms | nộp   0.02 ms
đuôi không giảm dài  | min hậu tố   1.04 ms | quét   1.05 ms | bisect   1.04 ms | nộp   1.04 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), bộ **kiểm tra tính hợp lệ** độc lập với thuật toán, test ngẫu nhiên có seed và test cỡ lớn.

In [19]:
def is_valid_and_best(arr: List[int], res: List[int]) -> bool:
    """Kiểm tra độc lập: res là arr hoặc là kết quả của đúng một lần đổi chỗ, nhỏ hơn arr, và tối ưu."""
    if sorted(res) != sorted(arr) or len(res) != len(arr):
        return False
    if res == arr:
        return all(not (arr[j] < arr[i]) for i, j in combinations(range(len(arr)), 2))   # không có cách đổi nào làm nhỏ hơn
    diff = [k for k in range(len(arr)) if arr[k] != res[k]]
    if len(diff) != 2 or not res < arr:
        return False
    i, j = diff
    if not (arr[i] == res[j] and arr[j] == res[i]):
        return False
    return res == prev_brute(arr)

all_impls = {"min hậu tố": sol_min, "quét": sol_scan, "bisect": sol_bisect, "nộp": sol,
             "prev_suffix_min": prev_suffix_min, "prev_scan": prev_scan, "prev_bisect": prev_bisect}

def check_all(arr, expected=None, brute=True):
    results = {name: fn(arr[:]) for name, fn in all_impls.items()}
    first = results["nộp"]
    for name, got in results.items():
        assert got == first, (arr[:10], name, got[:10], first[:10])
    if brute and len(arr) <= 60:
        assert first == prev_brute(arr), arr
        assert is_valid_and_best(arr, first), arr
    if expected is not None:
        assert first == expected, (arr, first, expected)
    return first

# 12.1 ví dụ đề
for arr, expected in examples:
    check_all(arr, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], [1]),                              # một phần tử: không có cặp
    ([2, 2], [2, 2]),                        # hai phần tử bằng nhau: đổi không đổi gì, không nhỏ hơn
    ([1, 2], [1, 2]),                        # tăng: đã nhỏ nhất
    ([2, 1], [1, 2]),                        # hai phần tử giảm: đổi cho nhỏ hơn
    ([3, 2, 1], [3, 1, 2]),                  # i = 1 (cuối), không phải i = 0
    ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),      # không giảm
    ([5, 5, 5, 5], [5, 5, 5, 5]),            # toàn bằng nhau
    ([3, 1, 1], [1, 3, 1]),                  # trùng: lấy bản trái nhất
    ([5, 1, 3, 3, 3], [3, 1, 5, 3, 3]),      # giá trị lớn nhất < 5 là 3; lấy bản trái nhất (vị trí 2)
    ([1, 3, 3, 2], [1, 3, 2, 3]),            # i = 2 (3 > 2); j = 3
    ([4, 5, 5, 3, 3], [4, 5, 3, 5, 3]),      # i = 2 (5 > 3), j trái nhất của 3 là vị trí 3
    ([2, 1, 3, 3, 1], [2, 1, 3, 1, 3]),      # i = 3 (3 > 1)
    ([10000, 1], [1, 10000]),                # biên giá trị
    ([1, 10000, 1], [1, 1, 10000]),          # i = 1: đổi với phần tử cuối
    ([9, 8, 7, 6, 5], [9, 8, 7, 5, 6]),      # giảm dần: đổi hai phần tử cuối
]
for arr, expected in edge:
    check_all(arr, expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 hàm kiểm tra không bị "nhầm": phải từ chối kết quả sai
assert not is_valid_and_best([3, 2, 1], [3, 2, 1])          # đáng lẽ phải đổi
assert not is_valid_and_best([3, 2, 1], [2, 3, 1])          # nhỏ hơn nhưng chưa tối ưu
assert not is_valid_and_best([1, 2, 3], [1, 3, 2])          # lớn hơn arr
assert not is_valid_and_best([1, 2, 3], [1, 2, 4])          # không phải hoán vị
assert is_valid_and_best([3, 2, 1], [3, 1, 2])
print("12.3 hàm kiểm tra: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 15 ca
12.3 hàm kiểm tra: OK


In [20]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_first_i(arr):                         # chọn i đầu tiên (trái nhất) thay vì cuối cùng
    arr = arr[:]
    n = len(arr)
    for i in range(n - 1):
        if arr[i] > arr[i + 1]:
            j = max((k for k in range(i + 1, n) if arr[k] < arr[i]), key=lambda k: (arr[k], -k))
            arr[i], arr[j] = arr[j], arr[i]
            return arr
    return arr

def wrong_smallest_j(arr):                      # chọn giá trị nhỏ nhất bên phải thay vì lớn nhất < arr[i]
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    j = min(range(i + 1, n), key=lambda k: (arr[k], k))
    arr[i], arr[j] = arr[j], arr[i]
    return arr

def wrong_rightmost_j(arr):                     # bỏ quy tắc trái nhất: lấy bản bên phải nhất
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    j = n - 1
    while arr[j] >= arr[i]:
        j -= 1
    arr[i], arr[j] = arr[j], arr[i]
    return arr

def wrong_adjacent(arr):                        # chỉ đổi với phần tử kề bên
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    arr[i], arr[i + 1] = arr[i + 1], arr[i]
    return arr

def wrong_allow_equal(arr):                     # dùng >= thay vì >: coi mảng bằng nhau là "có thể giảm"
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] < arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    j = n - 1
    while arr[j] > arr[i]:
        j -= 1
    while j > i + 1 and arr[j - 1] == arr[j]:
        j -= 1
    arr[i], arr[j] = arr[j], arr[i]
    return arr

def wrong_empty(arr):                           # trả về [] khi không làm được
    res = prev_scan(arr)
    return [] if res == arr else res

def find_counter(wrong, tries=40000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 7)
        arr = [rng.randint(1, 4) for _ in range(n)]
        if wrong(arr) != prev_brute(arr):
            return arr
    return None

counters = {}
for name, fn in (("chọn i đầu tiên", wrong_first_i), ("chọn giá trị nhỏ nhất", wrong_smallest_j),
                 ("chọn bản trùng bên phải nhất", wrong_rightmost_j), ("chỉ đổi phần tử kề", wrong_adjacent),
                 ("dùng >= (đổi phần tử bằng nhau)", wrong_allow_equal), ("trả về [] khi không làm được", wrong_empty)):
    arr = find_counter(fn)
    assert arr is not None, name
    counters[name] = arr
    print(f"{name:34} arr={arr}: sai -> {fn(arr)}, đúng -> {prev_brute(arr)}")
    check_all(arr)
print("12.4 phản ví dụ: OK")

chọn i đầu tiên                    arr=[1, 3, 4, 1, 3, 1, 4]: sai -> [1, 3, 3, 1, 4, 1, 4], đúng -> [1, 3, 4, 1, 1, 3, 4]
chọn giá trị nhỏ nhất              arr=[1, 3, 2, 2, 4, 1, 3]: sai -> [1, 3, 2, 2, 1, 4, 3], đúng -> [1, 3, 2, 2, 3, 1, 4]
chọn bản trùng bên phải nhất       arr=[3, 3, 2, 2]: sai -> [3, 2, 2, 3], đúng -> [3, 2, 3, 2]
chỉ đổi phần tử kề                 arr=[1, 3, 2, 2, 4, 1, 3]: sai -> [1, 3, 2, 2, 1, 4, 3], đúng -> [1, 3, 2, 2, 3, 1, 4]
dùng >= (đổi phần tử bằng nhau)    arr=[3, 3, 2, 2]: sai -> [3, 3, 2, 2], đúng -> [3, 2, 3, 2]
trả về [] khi không làm được       arr=[4, 4]: sai -> [], đúng -> [4, 4]
12.4 phản ví dụ: OK


In [21]:
# 12.5 ngẫu nhiên nhỏ: so oracle và hàm kiểm tra (mọi dải giá trị)
rng = random.Random(4053)
for _ in range(1500):
    n = rng.randint(1, 12)
    hi = rng.choice([2, 3, 5, 10, 10 ** 4])
    arr = [rng.randint(1, hi) for _ in range(n)]
    check_all(arr)
print("12.5 ngẫu nhiên nhỏ: OK")

# 12.6 ngẫu nhiên vừa (n <= 60) và lớn (n tới 10^4: so các cách nhanh với nhau và với hàm kiểm tra theo định nghĩa i)
for _ in range(60):
    n = rng.randint(30, 60)
    check_all([rng.randint(1, rng.choice([3, 20, 10 ** 4])) for _ in range(n)])
for _ in range(20):
    n = rng.randint(5000, 10 ** 4)
    arr = [rng.randint(1, rng.choice([3, 50, 10 ** 4])) for _ in range(n)]
    res = check_all(arr, brute=False)
    assert sorted(res) == sorted(arr) and (res == arr or res < arr)
    diff = [k for k in range(n) if arr[k] != res[k]]
    assert res == arr or (len(diff) == 2 and arr[diff[0]] == res[diff[1]] and arr[diff[1]] == res[diff[0]])
print("12.6 ngẫu nhiên vừa và lớn: OK")

# 12.7 tính chất: kết quả là hoán vị, nhỏ hơn hoặc bằng, đổi lặp lại giảm dần tới hoán vị nhỏ nhất, hàm không làm đổi bản gốc khi dùng bản sao
for _ in range(150):
    n = rng.randint(1, 6)
    arr = [rng.randint(1, 4) for _ in range(n)]
    cur, steps = arr[:], 0
    while True:
        nxt = sol(cur)
        assert sorted(nxt) == sorted(arr)
        if nxt == cur:
            break
        assert nxt < cur
        cur, steps = nxt, steps + 1
        assert steps < 10 ** 5
    assert cur == sorted(cur)                      # điểm dừng là hoán vị không giảm
    orig = arr[:]
    sol(arr)
    assert arr == orig
print("12.7 tính chất: OK")

12.5 ngẫu nhiên nhỏ: OK


12.6 ngẫu nhiên vừa và lớn: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chọn `i` đầu tiên (trái nhất) thay vì cuối cùng

Chọn vị trí đổi sớm làm tiền tố giữ nguyên ngắn, kết quả bị nhỏ hơn mức có thể.

In [22]:
arr = counters["chọn i đầu tiên"]
print("arr =", arr, "| sai ->", wrong_first_i(arr), "| đúng ->", sol(arr))
assert wrong_first_i(arr) != sol(arr)

arr = [1, 3, 4, 1, 3, 1, 4] | sai -> [1, 3, 3, 1, 4, 1, 4] | đúng -> [1, 3, 4, 1, 1, 3, 4]


Vì sao sai: kết quả nhỏ hơn mức tối ưu vì thay đổi sớm làm giảm một vị trí ưu tiên cao. **Đúng:** `i` là vị trí **cuối cùng** có `arr[i] > arr[i + 1]`.

### Lỗi 2 — Chọn giá trị nhỏ nhất thay vì lớn nhất nhỏ hơn `arr[i]`

Ta cần hoán vị **lớn nhất** trong các hoán vị nhỏ hơn: giá trị đưa vào `i` phải sát `arr[i]` nhất.

In [23]:
arr = counters["chọn giá trị nhỏ nhất"]
print("arr =", arr, "| sai ->", wrong_smallest_j(arr), "| đúng ->", sol(arr))
assert wrong_smallest_j(arr) != sol(arr)

arr = [1, 3, 2, 2, 4, 1, 3] | sai -> [1, 3, 2, 2, 1, 4, 3] | đúng -> [1, 3, 2, 2, 3, 1, 4]


Vì sao sai: giá trị nhỏ hơn làm kết quả nhỏ hơn mức cần. **Đúng:** lùi `j` từ cuối cho tới khi `arr[j] < arr[i]` (hậu tố không giảm nên đó là giá trị lớn nhất `< arr[i]`).

### Lỗi 3 — Quên quy tắc "trái nhất" khi có phần tử trùng

In [24]:
arr = counters["chọn bản trùng bên phải nhất"]
print("arr =", arr, "| sai ->", wrong_rightmost_j(arr), "| đúng ->", sol(arr))
assert wrong_rightmost_j(arr) != sol(arr)

arr = [3, 3, 2, 2] | sai -> [3, 2, 2, 3] | đúng -> [3, 2, 3, 2]


Vì sao sai: đặt `arr[i]` (lớn hơn `v`) vào vị trí **sớm** hơn cho kết quả lớn hơn. **Đúng:** thêm vòng `while arr[j - 1] == arr[j]: j -= 1`.

### Lỗi 4 — Chỉ đổi với phần tử kề bên

In [25]:
arr = counters["chỉ đổi phần tử kề"]
print("arr =", arr, "| sai ->", wrong_adjacent(arr), "| đúng ->", sol(arr))
assert wrong_adjacent(arr) != sol(arr)

arr = [1, 3, 2, 2, 4, 1, 3] | sai -> [1, 3, 2, 2, 1, 4, 3] | đúng -> [1, 3, 2, 2, 3, 1, 4]


Vì sao sai: phần tử cần đổi có thể ở xa (ví dụ `[1,9,4,6,7]` đổi `9` với `7`). **Đúng:** tìm `j` trong cả hậu tố.

### Lỗi 5 — Dùng `>=` làm đổi cho phần tử bằng nhau, hoặc trả về `[]`

`arr[i] >= arr[i + 1]` coi mảng có phần tử bằng nhau là "có thể giảm"; đổi hai phần tử bằng nhau không làm mảng nhỏ hơn. Và khi không làm được đề yêu cầu trả về **chính mảng**, không phải `[]`.

In [26]:
for name, fn in (("dùng >= (đổi phần tử bằng nhau)", wrong_allow_equal), ("trả về [] khi không làm được", wrong_empty)):
    arr = counters[name]
    print(f"{name}: arr = {arr} | sai -> {fn(arr)} | đúng -> {sol(arr)}")
    assert fn(arr) != sol(arr)

dùng >= (đổi phần tử bằng nhau): arr = [3, 3, 2, 2] | sai -> [3, 3, 2, 2] | đúng -> [3, 2, 3, 2]
trả về [] khi không làm được: arr = [4, 4] | sai -> [] | đúng -> [4, 4]


Vì sao sai: chỉ được đổi khi `arr[i] > arr[j]` thật sự; **Đúng:** điều kiện lùi `arr[i] <= arr[i + 1]` và `arr[j] >= arr[i]`, và `return arr` khi `i < 0`.

### Lỗi 6 — Quên rằng hàm sửa mảng **tại chỗ**

Cách nộp sửa chính `arr`. Nếu muốn so sánh kết quả với bản gốc trong test, phải sao chép trước, nếu không `kết quả < arr` luôn sai vì chúng là **cùng một danh sách**.

In [27]:
arr = [1, 9, 4, 6, 7]
res = Solution().prevPermOpt1(arr)                 # sửa tại chỗ
print("res =", res, "| arr =", arr, "| res is arr:", res is arr, "| res < arr:", res < arr)
assert res is arr and not (res < arr)

arr = [1, 9, 4, 6, 7]
res = Solution().prevPermOpt1(arr[:])              # truyền bản sao
print("res =", res, "| arr =", arr, "| res < arr:", res < arr)
assert res < arr

res = [1, 7, 4, 6, 9] | arr = [1, 7, 4, 6, 9] | res is arr: True | res < arr: False
res = [1, 7, 4, 6, 9] | arr = [1, 9, 4, 6, 7] | res < arr: True


Vì sao sai: so sánh một danh sách với chính nó luôn là `False`. **Đúng:** truyền bản sao `arr[:]` khi cần giữ bản gốc để kiểm tra.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "hoán vị lớn nhất nhỏ hơn / nhỏ nhất lớn hơn, với ít thao tác đổi chỗ" ⇒ tham lam theo **vị trí khác biệt đầu tiên**: càng về bên phải càng tốt.
- Quét **từ phải sang trái** để tìm điểm "vi phạm" thứ tự: cùng ý tưởng với **Next Permutation** (31).
- Khi hậu tố đã sắp xếp, dùng `bisect` thay vì quét thêm.
- Với giá trị trùng, nghĩ xem đặt giá trị lớn hơn vào vị trí **sớm** hay **muộn** thì kết quả lớn hơn.
- Luôn kiểm bằng oracle: thử mọi cặp trên mảng nhỏ.
- Cùng họ: Next Permutation (31), Next Greater Element III (556), Maximum Swap (670).
- Test nên có: `n = 1`, mảng không giảm, giảm dần, toàn bằng nhau, nhiều phần tử trùng ở hậu tố, `i = 0` và `i = n - 2`.

## 15. Biến thể

1. **Hoán vị nhỏ nhất lớn hơn `arr` bằng một lần đổi** (đối xứng): đổi chiều mọi so sánh.
2. **Hoán vị trước đó đầy đủ** (không giới hạn một lần đổi): đảo ngược của Next Permutation (31).
3. **Tối đa `k` lần đổi chỗ.**
4. Bài liên quan: 31, 556, 670.

Kiểm biến thể 1 bằng oracle thử mọi cặp (viết tay độc lập):

In [28]:
def next_one_swap_brute(arr: List[int]) -> List[int]:
    n = len(arr)
    best = None
    for i in range(n):
        for j in range(i + 1, n):
            b = arr[:]
            b[i], b[j] = b[j], b[i]
            if b > arr and (best is None or b < best):
                best = b
    return best if best is not None else arr[:]

def next_one_swap(arr: List[int]) -> List[int]:
    arr = arr[:]
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] >= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    j = n - 1
    while arr[j] <= arr[i]:
        j -= 1
    while arr[j - 1] == arr[j]:
        j -= 1
    arr[i], arr[j] = arr[j], arr[i]
    return arr

rng = random.Random(15)
for _ in range(3000):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 8))]
    assert next_one_swap(a) == next_one_swap_brute(a), a
print("biến thể: hoán vị nhỏ nhất lớn hơn bằng một lần đổi: kiểm bằng oracle")

biến thể: hoán vị nhỏ nhất lớn hơn bằng một lần đổi: kiểm bằng oracle


Biến thể 2 (hoán vị trước đó đầy đủ) được dùng làm bài tập 5 và có đáp án ở mục 19.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | Theo định nghĩa | `O(n^3)` | `O(n)` | Oracle |
| 2. Min hậu tố | `i` theo `suffix_min`, quét `j` | `O(n)` | `O(n)` | Dễ hiểu |
| 3. Một lượt quét | Lùi `i`, lùi `j` | `O(n)` | `O(1)` | **Cách nộp chính** |
| 4. `bisect_left` | Tìm `j` bằng nhị phân | `O(n)` | `O(1)` | Khi muốn bỏ vòng lùi `j` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(Solution().prevPermOpt1([2, 1, 3]))
print(Solution().prevPermOpt1([1, 2, 3]))
print(Solution().prevPermOpt1([4, 1, 1, 3]))
```

In [29]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # [2, 1, 3]
my_guess_2 = None      # [1, 2, 3]
my_guess_3 = None      # [4, 1, 1, 3]

### Bài 2 — Tính tay (dễ-vừa)

`arr = [6, 2, 5, 5, 3, 3]`. Tìm `i`, `j` và kết quả. Chú ý giá trị trùng.

In [30]:
# Bài 2: điền i, j, kết quả
hand_i, hand_j, hand_result = None, None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách 3.

In [31]:
def exercise_3(arr: List[int]) -> List[int]:
    n = len(arr)
    i = n - 2
    # TODO 1: lùi i khi arr[i] <= arr[i + 1]
    if i < 0:
        return arr
    j = n - 1
    # TODO 2: lùi j khi arr[j] >= arr[i]
    # TODO 3: lùi j khi arr[j - 1] == arr[j]
    arr[i], arr[j] = arr[j], arr[i]
    return arr


# for a, expected in examples:
#     assert exercise_3(a[:]) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(arr)` dùng `bisect_left` trên hậu tố không giảm (trả về mảng mới, không sửa mảng vào).

In [32]:
def exercise_4(arr: List[int]) -> List[int]:
    # TODO
    pass


# for a, expected in examples:
#     assert exercise_4(a) == expected

### Bài 5 — Mini challenge: hoán vị đứng trước (khó)

Viết `exercise_5(arr)` trả về **hoán vị lớn nhất theo thứ tự từ điển mà nhỏ hơn `arr`** (không giới hạn số lần đổi), hoặc **chính `arr`** nếu `arr` đã là nhỏ nhất. Ví dụ:

```python
exercise_5([1, 3, 2])     -> [1, 2, 3]
exercise_5([3, 2, 1])     -> [3, 1, 2]
exercise_5([1, 1, 5])     -> [1, 1, 5]
exercise_5([2, 3, 1, 3, 3]) -> [2, 1, 3, 3, 3]
```
Gợi ý: như next permutation nhưng ngược dấu, và đoạn hậu tố phải được **sắp xếp giảm dần** (không chỉ đổi một lần).

In [33]:
def exercise_5(arr: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_5([1, 3, 2]) == [1, 2, 3]
# assert exercise_5([3, 2, 1]) == [3, 1, 2]
# assert exercise_5([1, 1, 5]) == [1, 1, 5]
# assert exercise_5([2, 3, 1, 3, 3]) == [2, 1, 3, 3, 3]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: tìm `i` cuối cùng có `arr[i] > arr[i + 1]`; nếu không có, trả về nguyên mảng.
- Mức 2: `[2,1,3]`: `i = 0` (`2 > 1`); hậu tố `[1,3]`, giá trị lớn nhất `< 2` là `1`.
- Mức 3: ba kết quả là `[1, 2, 3]`, `[1, 2, 3]`, `[3, 1, 1, 4]` (với `[4,1,1,3]`: `i = 0`, giá trị lớn nhất `< 4` là `3` ở vị trí `3`).

**Bài 2**
- Mức 1: tìm `i` cuối cùng có `arr[i] > arr[i + 1]`: `5 > 3` ở `i = 3`.
- Mức 2: hậu tố sau `i = 3` là `[3, 3]`; giá trị lớn nhất `< 5` là `3`, xuất hiện ở `j = 4` và `j = 5`.
- Mức 3: chọn bản trái nhất `j = 4`; kết quả `[6, 2, 5, 3, 5, 3]`.

**Bài 3**
- Mức 1: `while i >= 0 and arr[i] <= arr[i + 1]: i -= 1`.
- Mức 2: `while arr[j] >= arr[i]: j -= 1`.
- Mức 3: `while arr[j - 1] == arr[j]: j -= 1`.

**Bài 4**
- Mức 1: tìm `i` bằng vòng lùi như cách 3, chú ý sao chép mảng đầu vào.
- Mức 2: `p = bisect_left(a, a[i], i + 1, n)`.
- Mức 3: `v = a[p - 1]`; `j = bisect_left(a, v, i + 1, n)`; đổi `a[i]` với `a[j]`.

**Bài 5**
- Mức 1: tìm `i` cuối cùng có `arr[i] > arr[i + 1]` (như next permutation ngược).
- Mức 2: đổi `arr[i]` với giá trị lớn nhất `< arr[i]` ở hậu tố, rồi sắp xếp hậu tố **giảm dần**.
- Mức 3: `arr[i + 1:] = sorted(arr[i + 1:], reverse=True)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [34]:
r1, r2, r3 = sol([2, 1, 3]), sol([1, 2, 3]), sol([4, 1, 1, 3])
print(r1, r2, r3)
assert r1 == [1, 2, 3] and r2 == [1, 2, 3] and r3 == [3, 1, 1, 4]
assert (r1, r2, r3) == (prev_brute([2, 1, 3]), prev_brute([1, 2, 3]), prev_brute([4, 1, 1, 3]))

[1, 2, 3] [1, 2, 3] [3, 1, 1, 4]


### Đáp án Bài 2

`arr = [6, 2, 5, 5, 3, 3]`: `i = 3` là vị trí cuối cùng với `arr[i] > arr[i + 1]` (`5 > 3`). Hậu tố `arr[4:] = [3, 3]`; giá trị lớn nhất `< 5` là `3`, bản trái nhất ở `j = 4`. Kết quả `[6, 2, 5, 3, 5, 3]`.

In [35]:
hand_i_solution, hand_j_solution, hand_result_solution = 3, 4, [6, 2, 5, 3, 5, 3]
arr = [6, 2, 5, 5, 3, 3]
b = arr[:]
b[hand_i_solution], b[hand_j_solution] = b[hand_j_solution], b[hand_i_solution]
assert b == hand_result_solution == sol(arr) == prev_brute(arr)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [36]:
def exercise_3_solution(arr: List[int]) -> List[int]:
    n = len(arr)
    i = n - 2
    while i >= 0 and arr[i] <= arr[i + 1]:
        i -= 1
    if i < 0:
        return arr
    j = n - 1
    while arr[j] >= arr[i]:
        j -= 1
    while arr[j - 1] == arr[j]:
        j -= 1
    arr[i], arr[j] = arr[j], arr[i]
    return arr

for a, expected in examples:
    assert exercise_3_solution(a[:]) == expected
rng = random.Random(3)
for _ in range(500):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 8))]
    assert exercise_3_solution(a[:]) == prev_brute(a)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def exercise_4_solution(arr: List[int]) -> List[int]:
    a = arr[:]
    n = len(a)
    i = n - 2
    while i >= 0 and a[i] <= a[i + 1]:
        i -= 1
    if i < 0:
        return a
    p = bisect_left(a, a[i], i + 1, n)
    v = a[p - 1]
    j = bisect_left(a, v, i + 1, n)
    a[i], a[j] = a[j], a[i]
    return a

for a, expected in examples:
    assert exercise_4_solution(a) == expected
rng = random.Random(4)
for _ in range(500):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 8))]
    orig = a[:]
    assert exercise_4_solution(a) == prev_brute(a)
    assert a == orig
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Oracle độc lập: liệt kê mọi hoán vị phân biệt bằng `itertools.permutations`, lấy hoán vị lớn nhất nhỏ hơn `arr`.

In [38]:
def exercise_5_solution(arr: List[int]) -> List[int]:
    a = arr[:]
    n = len(a)
    i = n - 2
    while i >= 0 and a[i] <= a[i + 1]:
        i -= 1
    if i < 0:
        return a
    j = n - 1
    while a[j] >= a[i]:
        j -= 1
    a[i], a[j] = a[j], a[i]
    a[i + 1:] = sorted(a[i + 1:], reverse=True)
    return a

def prev_permutation_oracle(arr):
    smaller = [list(p) for p in set(permutations(arr)) if list(p) < arr]
    return max(smaller) if smaller else arr[:]

assert exercise_5_solution([1, 3, 2]) == [1, 2, 3]
assert exercise_5_solution([3, 2, 1]) == [3, 1, 2]
assert exercise_5_solution([1, 1, 5]) == [1, 1, 5]
assert exercise_5_solution([2, 3, 1, 3, 3]) == [2, 1, 3, 3, 3]
rng = random.Random(5)
for _ in range(500):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 7))]
    assert exercise_5_solution(a) == prev_permutation_oracle(a), a
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam từ phải sang trái)

```python
class Solution:
    def prevPermOpt1(self, arr: List[int]) -> List[int]:
        n = len(arr)
        i = n - 2
        while i >= 0 and arr[i] <= arr[i + 1]:           # i = vị trí cuối cùng có arr[i] > arr[i + 1]
            i -= 1
        if i < 0:
            return arr                                   # đã là hoán vị nhỏ nhất
        j = n - 1
        while arr[j] >= arr[i]:                          # giá trị lớn nhất < arr[i] trong hậu tố không giảm
            j -= 1
        while arr[j - 1] == arr[j]:                      # nhiều bản trùng: lấy bản trái nhất
            j -= 1
        arr[i], arr[j] = arr[j], arr[i]
        return arr
```

Công thức nhớ nhanh: **`i` cuối cùng có `arr[i] > arr[i+1]`; `j` là giá trị lớn nhất `< arr[i]` bên phải, lấy bản trái nhất; đổi.**

### Checklist trước khi submit

- [ ] `i` là vị trí **cuối cùng** (phải nhất) có `arr[i] > arr[i + 1]`?
- [ ] Nếu không có `i` thì trả về chính mảng (không phải `[]`)?
- [ ] `j` là giá trị lớn nhất **nhỏ hơn hẳn** `arr[i]` (dùng `>=` khi lùi `j`)?
- [ ] Với giá trị trùng, lấy `j` **trái nhất**?
- [ ] Biết rằng hàm sửa mảng tại chỗ; khi test thì truyền bản sao?
- [ ] Đã thử: `n = 1`, mảng không giảm, giảm dần, toàn bằng nhau, nhiều giá trị trùng, `n = 10^4`?